In [1]:
import os
import json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output
from dateutil import parser as dtparser

In [2]:
import os, sys
from pathlib import Path

# You are running the notebook inside: LIO/notebooks
# So LIO root is the parent folder.
HERE = Path.cwd()
SITE_ROOT = HERE.parent  # <-- THIS FIXES YOUR load_ssot path

SRC = SITE_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("CWD      :", HERE)
print("SITE_ROOT:", SITE_ROOT)
print("SRC      :", SRC)
print("config   :", SITE_ROOT / "etc" / "config.json")

from apm_core.ssot import load_ssot, list_sensors, get_sensor
from apm_core.settings import load_ini
from apm_core.db_interface import DBInterface

CWD      : /home/johnathanc/APM/LIO/notebooks
SITE_ROOT: /home/johnathanc/APM/LIO
SRC      : /home/johnathanc/APM/LIO/src
config   : /home/johnathanc/APM/LIO/etc/config.json


In [3]:
import logging

logger = logging.getLogger("TRAINING")
logger.setLevel(logging.INFO)
if not logger.handlers:
    h = logging.StreamHandler()
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(h)

logger.info(f"SITE_ROOT={SITE_ROOT}")

2026-02-24 13:40:43,981 | INFO | SITE_ROOT=/home/johnathanc/APM/LIO


In [4]:
ssot = load_ssot(SITE_ROOT)
debug = bool(ssot.get("debug", False))

settings = load_ini(SITE_ROOT, debug=debug)
db = DBInterface(settings=settings, logger=logger)

SENSORS = list_sensors(ssot)
SENSORS[:10], len(SENSORS)

2026-02-24 13:40:44,012 | INFO | PostgreSQL pool initialised
2026-02-24 13:40:44,028 | INFO | MSSQL connected | table=APM_Scalability_Dev


(['Mill1PinionConditionScore',
  'Mill1TrunnionConditionScore',
  'Mill1MotorConditionScore',
  'Mill1GearboxConditionScore'],
 4)

In [5]:
def trained_params_dir(site_root: Path, sensor_name: str, method: str) -> Path:
    p = site_root / "etc" / "trained_params" / sensor_name / method
    p.mkdir(parents=True, exist_ok=True)
    return p

In [6]:
import numpy as np
import pandas as pd
from datetime import datetime

def pull_wide_with_filter_tag(
    *,
    db: DBInterface,
    sensor_cfg: dict,
    start: datetime,
    end: datetime,
    method_name: str = "StatisticalThresholds",
) -> tuple[pd.DataFrame, pd.Series]:
    """
    Pulls feature tags + filter_tag.tag (if present), returns:
      df_wide: timestamp index, columns are raw tag strings (FEATURE tags only)
      section: raw filter tag series (same index)

    IMPORTANT: for TRAINING speed, prefers:
      sensor_cfg["Method"][method_name]["Training"]["pull_interval"]
    else falls back to:
      sensor_cfg["Other"]["granularity"] + ["granularity_type"]
    """

    features = sensor_cfg.get("Features", {}) or {}
    feature_tags = []
    for _, meta in features.items():
        if isinstance(meta, dict) and meta.get("tag"):
            feature_tags.append(meta["tag"])

    ft = sensor_cfg.get("filter_tag", {}) or {}
    filter_tag = ft.get("tag", None)

    tags = list(dict.fromkeys(feature_tags + ([filter_tag] if filter_tag else [])))

    # ---- TRAINING PULL INTERVAL (FAST PATH) ----
    interval_str = None
    try:
        method_block = (sensor_cfg.get("Method", {}) or {}).get(method_name, {}) or {}
        tr = (method_block.get("Training", {}) or {})
        interval_str = tr.get("pull_interval", None)
    except Exception:
        interval_str = None

    # ---- FALLBACK: runtime granularity ----
    if not interval_str:
        other = sensor_cfg.get("Other", {}) or {}
        n = int(other.get("granularity", 15))
        unit = str(other.get("granularity_type", "seconds")).lower()
        if unit in ("sec", "second", "seconds"):
            unit = "seconds"
        elif unit in ("min", "minute", "minutes"):
            unit = "minutes"
        elif unit in ("hr", "hour", "hours"):
            unit = "hours"
        interval_str = f"{n} {unit}"

    # Pull raw data (long format) from Postgres
    raw = db.pull_raw_tags_postgres(tags=tags, start=start, end=end, interval_str=interval_str)
    if raw is None or raw.empty:
        return pd.DataFrame(), pd.Series(dtype=float)

    df_long = raw[["rt_timestamp", "sourceidentifier", "rt_value"]].copy()
    df_long.rename(columns={"rt_timestamp": "timestamp", "sourceidentifier": "tag", "rt_value": "value"}, inplace=True)
    df_long["timestamp"] = pd.to_datetime(df_long["timestamp"])

    # Pivot to wide
    df_wide = df_long.pivot_table(index="timestamp", columns="tag", values="value", aggfunc="last").sort_index()

    # Extract filter/section series (and drop filter_tag from feature df)
    if filter_tag and filter_tag in df_wide.columns:
        section = pd.to_numeric(df_wide[filter_tag], errors="coerce").copy()
        df_wide = df_wide.drop(columns=[filter_tag])
    else:
        section = pd.Series(index=df_wide.index, data=np.nan, dtype=float)

    return df_wide, section


def exclude_off_and_startup(
    *,
    df_wide: pd.DataFrame,
    section_raw: pd.Series,
    filter_value: float,
    startup_minutes: int,
    smoothing_window: int = 10,
) -> pd.DataFrame:
    """
    Legacy-ish:
      - exclude section off (section < filter_value)
      - exclude startup window after off->on transition
      - rolling median smoothing + backfill
    """
    if df_wide is None or df_wide.empty:
        return pd.DataFrame()

    idx = df_wide.index
    section_raw = section_raw.reindex(idx)

    open_mask = (section_raw >= float(filter_value)).fillna(False)

    startup_mask = pd.Series(False, index=idx)
    if startup_minutes and int(startup_minutes) > 0:
        prev_open = open_mask.shift(1).fillna(False)
        transitions = (~prev_open) & (open_mask)
        for t in idx[transitions]:
            end = t + pd.Timedelta(minutes=int(startup_minutes))
            startup_mask.loc[(idx >= t) & (idx <= end)] = True

    exclude_mask = (~open_mask) | (startup_mask)
    clean = df_wide.loc[~exclude_mask].copy()

    if smoothing_window and smoothing_window > 1 and not clean.empty:
        clean = clean.rolling(int(smoothing_window)).median()
        clean = clean.bfill()

    return clean

In [7]:
def train_statistical_thresholds(
    *,
    ssot: dict,
    sensor_name: str,
    start: datetime,
    end: datetime,
) -> dict:
    sensor_cfg = ssot[sensor_name]
    ft = sensor_cfg.get("filter_tag", {}) or {}
    filter_value = float(ft.get("filter_value", 0.9))

    other = sensor_cfg.get("Other", {}) or {}
    startup_minutes = int(other.get("startup_period", 0))  # treat as minutes in new system
    smoothing_window = int(other.get("training_smoothing_window", 10))

    # Training params in SSOT:
    #   sensor.Method.StatisticalThresholds.Training.mode = "zscore" or "quantile"
    #   zscore_k, quantile_hi, resample_rule etc.
    st_block = ((sensor_cfg.get("Method", {}) or {}).get("StatisticalThresholds", {}) or {})
    tr = (st_block.get("Training", {}) or {})
    mode = str(tr.get("mode", "zscore")).lower()  # "zscore" or "quantile"
    z_k = float(tr.get("zscore_k", 3.0))
    q_hi = float(tr.get("quantile_hi", 0.99))
    q_lo = float(tr.get("quantile_lo", 1.0 - q_hi))
    resample_rule = str(tr.get("resample_rule", "h")).lower()  # e.g. "h"

    df_wide, section_raw = pull_wide_with_filter_tag(db=db, sensor_cfg=sensor_cfg, start=start, end=end, method_name="StatisticalThresholds")
    if df_wide.empty:
        return {"ok": False, "reason": "no data"}

    clean = exclude_off_and_startup(
        df_wide=df_wide,
        section_raw=section_raw,
        filter_value=filter_value,
        startup_minutes=startup_minutes,
        smoothing_window=smoothing_window,
    )
    if clean.empty:
        return {"ok": False, "reason": "all data excluded by off/startup rules"}

    # Resample like legacy (hourly medians)
    clean_rs = clean.resample(resample_rule).median().dropna(how="all")
    if clean_rs.empty:
        return {"ok": False, "reason": "empty after resample"}

    # Build thresholds per raw tag
    thresholds = {"high": {}, "low": {}, "stats": {}}

    for tag in clean_rs.columns:
        s = pd.to_numeric(clean_rs[tag], errors="coerce").dropna()
        if s.empty:
            continue

        if mode == "quantile":
            hi = float(s.quantile(q_hi))
            lo = float(s.quantile(q_lo))
            mu = float(s.mean())
            sd = float(s.std(ddof=1)) if s.size > 1 else 0.0
        else:
            mu = float(s.mean())
            sd = float(s.std(ddof=1)) if s.size > 1 else 0.0
            hi = mu + z_k * sd
            lo = mu - z_k * sd

        thresholds["high"][tag] = hi
        thresholds["low"][tag] = lo
        thresholds["stats"][tag] = {"mean": mu, "std": sd, "n": int(s.size)}

    out = {
        "ok": True,
        "sensor": sensor_name,
        "method": "StatisticalThresholds",
        "trained_at_utc": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "window": {"start": start.isoformat(), "end": end.isoformat()},
        "training": {
            "mode": mode,
            "zscore_k": z_k,
            "quantile_hi": q_hi,
            "quantile_lo": q_lo,
            "resample_rule": resample_rule,
            "startup_minutes": startup_minutes,
            "filter_value": filter_value,
            "smoothing_window": smoothing_window,
        },
        "thresholds": thresholds,
    }
    return out

In [8]:
def preview_thresholds(sensor_cfg: dict, trained: dict, n_tags: int = 6):
    thr = trained.get("thresholds", {})
    hi_map = thr.get("high", {})
    lo_map = thr.get("low", {})

    tags = list(hi_map.keys())[:n_tags]
    if not tags:
        print("No thresholds to preview.")
        return

    fig, axes = plt.subplots(len(tags), 1, figsize=(14, 3 * len(tags)), sharex=True)
    if len(tags) == 1:
        axes = [axes]

    # Pull a small recent window for plotting
    end = dtparser.parse(trained["window"]["end"])
    start = end - timedelta(hours=48)

    df_wide, section_raw = pull_wide_with_filter_tag(db=db, sensor_cfg=sensor_cfg, start=start, end=end)
    if df_wide.empty:
        print("No plot data.")
        return

    for ax, tag in zip(axes, tags):
        if tag not in df_wide.columns:
            continue
        s = pd.to_numeric(df_wide[tag], errors="coerce")
        ax.plot(s.index, s.values, label=tag)
        ax.axhline(hi_map[tag], linestyle="--", label="High")
        ax.axhline(lo_map[tag], linestyle="--", label="Low")
        ax.grid(True)
        ax.legend(loc="upper right")

    plt.show()

In [9]:
sensor_dd = widgets.Dropdown(
    options=["ALL"] + SENSORS,
    value="ALL",
    description="Sensor:",
    layout=widgets.Layout(width="450px")
)

start_txt = widgets.Text(
    value=(datetime.utcnow() - pd.Timedelta(days=90)).strftime("%Y-%m-%d %H:%M:%S"),
    description="Start (UTC):",
    layout=widgets.Layout(width="450px")
)

end_txt = widgets.Text(
    value=datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
    description="End (UTC):",
    layout=widgets.Layout(width="450px")
)

train_btn = widgets.Button(description="Train StatisticalThresholds", button_style="success")
out_box = widgets.Output()

display(widgets.VBox([sensor_dd, start_txt, end_txt, train_btn, out_box]))

In [10]:
def _parse_dt(s: str) -> datetime:
    return dtparser.parse(s)

def on_train_clicked(_):
    with out_box:
        clear_output()
        sel = sensor_dd.value
        start = _parse_dt(start_txt.value)
        end = _parse_dt(end_txt.value)

        targets = SENSORS if sel == "ALL" else [sel]
        print(f"Training StatisticalThresholds for: {targets}")
        print(f"Window UTC: {start} -> {end}\n")

        for sn in targets:
            trained = train_statistical_thresholds(ssot=ssot, sensor_name=sn, start=start, end=end)
            if not trained.get("ok", False):
                print(f"[{sn}] FAILED: {trained.get('reason')}")
                continue

            p = trained_params_dir(SITE_ROOT, sn, "StatisticalThresholds") / "trained_thresholds.json"
            with open(p, "w", encoding="utf-8") as f:
                json.dump(trained, f, indent=2)

            print(f"[{sn}] OK -> saved: {p}")

train_btn.on_click(on_train_clicked)